<div style='background:linear-gradient(135deg,#0f2942,#27ae60);padding:28px;border-radius:14px;margin-bottom:8px'>
<h1 style='color:white;margin:0;font-size:28px'>🏥 Como Conversar com os Seus Dados</h1>
<p style='color:#d5f5e3;margin:10px 0 4px;font-size:15px'>IDASH 2026 · CNES Leitos Hospitalares · Brasil Jun/2025</p>
<p style='color:rgba(255,255,255,.5);margin:0;font-size:13px'>GPT-4o-mini · RAG com agregações · Gradio · NotebookLM</p>
</div>

---

## 🗺️ Estrutura da Aula

| Bloco | Etapa | Células | No Colab |
|---|---|---|---|
| **1** | Preparar o Ambiente | 1 e 2 | ✅ |
| **2** | Carregar os Dados | 3 e 4 | ✅ |
| **3** | Explorar os Dados | 5 e 6 | ✅ |
| **3B** | Validar com SQL Workbench | — | ❌ externo |
| **4** | Entender o RAG | conceito | ✅ |
| **5** | Conectar a IA | 7 | ✅ |
| **6** | Motor do Chat | 8 e 9 | ✅ |
| **7** | Interface Gradio | 10 | ✅ |
| **8** | Exportar Resultados | 11 | ✅ |

> ⚠️ **Ordem obrigatória:** execute os blocos de cima para baixo.
> Se o Colab reconectar: **Ambiente de execução → Executar tudo** (`Ctrl+F9`).


---
## 📦 BLOCO 1 — Preparar o Ambiente
> Instalamos **5 bibliotecas** e configuramos o ambiente Python para toda a sessão.
---

In [ ]:
%%capture
# ============================================================
# CÉLULA 1 — Instalar bibliotecas
# ============================================================
# pandas    → manipular tabelas de dados
# numpy     → cálculos numéricos
# matplotlib → criar gráficos
# seaborn   → estilo visual dos gráficos
# openai    → conectar ao GPT-4o-mini
# gradio    → interface visual do chatbot
# ============================================================
!pip install openai gradio pandas numpy matplotlib seaborn

print('✅ 5 bibliotecas instaladas: pandas, numpy, matplotlib, seaborn, openai, gradio')

In [ ]:
# ============================================================
# CÉLULA 2 — Importar e configurar o ambiente
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('✅ Bibliotecas carregadas!')
print(f'   pandas {pd.__version__} | numpy {np.__version__}')

---
## 📂 BLOCO 2 — Carregar os Dados
> Upload do CSV e enriquecimento: criamos as colunas `uf_sg` e `regiao` a partir do `codufmun`.
---

In [ ]:
# ============================================================
# CÉLULA 3 — Upload e leitura do CSV (padrão DATASUS/Brasil)
# ============================================================
# Padrão brasileiro:
#   Separador : ;  (ponto e vírgula)
#   Decimal   : ,  (vírgula)
#   Encoding  : latin-1  (acentos: ç, ã, é)
#   Data      : DD/MM/AAAA
# Fontes: CNES, SINAN, SIM, IBGE, e-SUS
# ============================================================
from google.colab import files

print('📤 Selecione o arquivo CnesLeitosBR.csv')
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(
    nome_arquivo,
    sep=';',             # separador: ponto e vírgula
    encoding='latin-1',  # suporte a acentos brasileiros
    decimal=',',         # decimal: vírgula (padrão BR)
    thousands='.',       # milhar: ponto (1.234 = mil)
    dayfirst=True,       # datas: DD/MM/AAAA
    on_bad_lines='skip'  # pular linhas com problema
)

# Corrigir BOM e espaços nos nomes de colunas
df.columns = [c.replace('ï»¿', '').strip() for c in df.columns]

print(f'\n✅ Arquivo carregado: {nome_arquivo}')
print(f'📐 {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'📊 Colunas: {list(df.columns)}')
df.head(3)

In [ ]:
# ============================================================
# CÉLULA 4 — Enriquecer os dados (derivar UF e Região)
# ============================================================
# O CSV bruto tem apenas 'codufmun'. Criamos:
#   uf_sg  → sigla do estado (ex: '35' → 'SP')
#   regiao → Norte / Nordeste / CO / Sudeste / Sul
# ============================================================

df['uf_co'] = df['codufmun'].astype(str).str[:2]

uf_map = {
    '11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO',
    '21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL','28':'SE','29':'BA',
    '31':'MG','32':'ES','33':'RJ','35':'SP',
    '41':'PR','42':'SC','43':'RS',
    '50':'MS','51':'MT','52':'GO','53':'DF'
}
df['uf_sg'] = df['uf_co'].map(uf_map)

reg_map = {
    'Norte':        ['AM','PA','AC','RR','RO','AP','TO'],
    'Nordeste':     ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Centro-Oeste': ['MT','MS','GO','DF'],
    'Sudeste':      ['SP','RJ','MG','ES'],
    'Sul':          ['PR','SC','RS']
}
uf_to_reg = {uf: r for r, ufs in reg_map.items() for uf in ufs}
df['regiao'] = df['uf_sg'].map(uf_to_reg)

# Garantir colunas numéricas
for col in ['qt_exist', 'qt_sus', 'qt_nsus']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Calcular qt_nsus se não existir no CSV
if 'qt_nsus' not in df.columns:
    df['qt_nsus'] = df['qt_exist'] - df['qt_sus']

print('✅ Base enriquecida!')
print(f'🗺️  UFs: {df["uf_sg"].nunique()} | Regiões: {df["regiao"].nunique()}')
print(f'🏥 Leitos totais : {df["qt_exist"].sum():,.0f}')
print(f'💚 Leitos SUS    : {df["qt_sus"].sum():,.0f} ({df["qt_sus"].sum()/df["qt_exist"].sum()*100:.1f}%)')
print(f'📋 Tipos de leito: {df["DS_TP_LEITO"].nunique() if "DS_TP_LEITO" in df.columns else "coluna não encontrada"}')

---
## 📊 BLOCO 3 — Explorar os Dados (EDA)
> A IA só responde bem quando recebe contexto correto. EDA revela lacunas e a estrutura real dos dados antes de alimentar a IA.
---

In [ ]:
# ============================================================
# CÉLULA 5 — Resumo geral dos dados
# ============================================================
lt_e = int(df['qt_exist'].sum())
lt_s = int(df['qt_sus'].sum())
lt_n = int(df['qt_nsus'].sum())
df_uti = df[df['DS_CO_LEITO'].str.contains('UTI', na=False)] if 'DS_CO_LEITO' in df.columns else pd.DataFrame()
uti_e  = int(df_uti['qt_exist'].sum()) if len(df_uti) > 0 else 0
uti_s  = int(df_uti['qt_sus'].sum())   if len(df_uti) > 0 else 0

print('=' * 55)
print('RESUMO — CNES LEITOS HOSPITALARES BRASIL Jun/2025')
print('=' * 55)
print(f'  Registros na base    : {len(df):>10,}')
print(f'  Leitos existentes    : {lt_e:>10,}')
print(f'  Leitos SUS           : {lt_s:>10,}  ({lt_s/lt_e*100:.1f}%)')
print(f'  Leitos não-SUS       : {lt_n:>10,}  ({lt_n/lt_e*100:.1f}%)')
print(f'  UTI existentes       : {uti_e:>10,}')
print(f'  UTI SUS              : {uti_s:>10,}')
print(f'  UFs                  : {df["uf_sg"].nunique():>10}')
print(f'  Tipos de leito       : {df["DS_TP_LEITO"].nunique():>10}')
print(f'  Especialidades       : {df["DS_CO_LEITO"].nunique():>10}')

# Missings
miss = (df.isnull().sum()/len(df)*100).round(1)
miss = miss[miss > 0].sort_values(ascending=False)
print(f'\n  Colunas com dados ausentes: {len(miss)}')
if len(miss) > 0:
    for col, pct in miss.head(5).items():
        print(f'    {col}: {pct}%')

In [ ]:
# ============================================================
# CÉLULA 6 — 3 gráficos essenciais
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('CNES Leitos Hospitalares — Brasil Jun/2025', fontsize=14, fontweight='bold')

# Gráfico 1: Leitos por UF
uf_dist = df.groupby('uf_sg')['qt_exist'].sum().sort_values(ascending=False)
cores = ['#e74c3c' if uf in uf_dist.head(5).index else '#3498db' for uf in uf_dist.index]
axes[0].bar(uf_dist.index, uf_dist.values, color=cores, edgecolor='white')
axes[0].set_title('Leitos por UF (🔴 Top 5)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=90, labelsize=7)

# Gráfico 2: % SUS por Região
reg_data = df.groupby('regiao').agg(exist=('qt_exist','sum'), sus=('qt_sus','sum'))
reg_data['pct_sus'] = (reg_data['sus']/reg_data['exist']*100).round(1)
reg_data = reg_data.reindex([r for r in ['Norte','Nordeste','Centro-Oeste','Sudeste','Sul'] if r in reg_data.index])
cores_r = ['#2ecc71' if v >= 60 else '#f39c12' if v >= 50 else '#e74c3c' for v in reg_data['pct_sus']]
axes[1].bar(reg_data.index, reg_data['pct_sus'], color=cores_r, edgecolor='white')
axes[1].axhline(lt_s/lt_e*100, color='black', linestyle='--', linewidth=1.5, label=f'Média BR ({lt_s/lt_e*100:.1f}%)')
axes[1].set_title('% Leitos SUS por Região', fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)
axes[1].tick_params(axis='x', rotation=15)

# Gráfico 3: Top 10 Especialidades
if 'DS_CO_LEITO' in df.columns:
    top_esp = df.groupby('DS_CO_LEITO')['qt_exist'].sum().sort_values(ascending=False).head(10)
    axes[2].barh(top_esp.index[::-1], top_esp.values[::-1], color='#9b59b6', edgecolor='white')
    axes[2].set_title('Top 10 Especialidades', fontweight='bold')
    axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

---
## 🗄️ BLOCO 3B — Validar com SQL Workbench
> ⚠️ **Esta etapa é feita fora do Colab**, no navegador em [sql-workbench.com](https://sql-workbench.com).
> Arraste o arquivo `CnesLeitosBR.csv` para o Workbench e execute as consultas abaixo para confirmar que os números do Python estão corretos.

### Consulta 1 — Totais gerais
```sql
SELECT
    COUNT(*)                                        AS total_registros,
    SUM(qt_exist)                                   AS total_leitos,
    SUM(qt_sus)                                     AS total_sus,
    SUM(qt_exist) - SUM(qt_sus)                    AS total_nao_sus,
    ROUND(SUM(qt_sus) * 100.0 / SUM(qt_exist), 1) AS pct_sus
FROM "CnesLeitosBR.csv";
```

### Consulta 2 — Leitos por UF (Top 10)
> ⚠️ A coluna `uf_sg` não existe no CSV bruto — derivamos do `codufmun` diretamente no SQL.
```sql
SELECT
    CASE LEFT(CAST(codufmun AS VARCHAR), 2)
        WHEN '11' THEN 'RO' WHEN '12' THEN 'AC' WHEN '13' THEN 'AM'
        WHEN '14' THEN 'RR' WHEN '15' THEN 'PA' WHEN '16' THEN 'AP'
        WHEN '17' THEN 'TO' WHEN '21' THEN 'MA' WHEN '22' THEN 'PI'
        WHEN '23' THEN 'CE' WHEN '24' THEN 'RN' WHEN '25' THEN 'PB'
        WHEN '26' THEN 'PE' WHEN '27' THEN 'AL' WHEN '28' THEN 'SE'
        WHEN '29' THEN 'BA' WHEN '31' THEN 'MG' WHEN '32' THEN 'ES'
        WHEN '33' THEN 'RJ' WHEN '35' THEN 'SP' WHEN '41' THEN 'PR'
        WHEN '42' THEN 'SC' WHEN '43' THEN 'RS' WHEN '50' THEN 'MS'
        WHEN '51' THEN 'MT' WHEN '52' THEN 'GO' WHEN '53' THEN 'DF'
    END                                             AS uf_sg,
    SUM(qt_exist)                                   AS leitos_existentes,
    SUM(qt_sus)                                     AS leitos_sus,
    ROUND(SUM(qt_sus) * 100.0 / SUM(qt_exist), 1) AS pct_sus
FROM "CnesLeitosBR.csv"
GROUP BY LEFT(CAST(codufmun AS VARCHAR), 2)
ORDER BY leitos_existentes DESC
LIMIT 10;
```

### Consulta 3 — Inconsistências lógicas
```sql
SELECT
    COUNT(*)                   AS registros_inconsistentes,
    SUM(qt_sus - qt_exist)     AS diferenca_total,
    MAX(qt_sus - qt_exist)     AS maior_diferenca
FROM "CnesLeitosBR.csv"
WHERE qt_sus > qt_exist;
```
> Se `registros_inconsistentes = 0` → ✅ base consistente  
> Se `registros_inconsistentes > 0` → ⚠️ há erros no cadastro CNES (comum em dados reais)
---

---
## 🧠 BLOCO 4 — Entender o RAG
> Antes de codar, entendemos a estratégia que faz a IA responder com dados reais.

### 🔍 RAG (Retrieval-Augmented Generation)
```
Pergunta → Roteador → Seleciona tabelas relevantes → LLM responde com contexto
```

### ⚖️ RAG vs Fine-tuning

| Critério | RAG ✅ | Fine-tuning |
|---|---|---|
| Atualiza com novo CSV | ✅ Instantâneo | ❌ Retreina tudo |
| Custo inicial | ✅ Baixo | ❌ Alto (GPU) |
| Transparente | ✅ Cita fontes | ❌ Opaco |
| Para dados do SUS | ✅ **Recomendado** | 🟡 Casos específicos |

> **Nossa escolha:** RAG com agregações pré-calculadas — simples, rápido, auditável.
---

---
## 🔌 BLOCO 5 — Conectar a IA
> 🔐 A API Key não é salva — some quando o runtime fechar. Nunca compartilhe em prints ou notebooks públicos.
---

In [ ]:
# ============================================================
# CÉLULA 7 — Conectar à API OpenAI
# ============================================================
import openai
from getpass import getpass

# getpass esconde o que você digita (como campo de senha)
OPENAI_API_KEY = getpass('🔑 Cole sua OpenAI API Key: ').strip()
client_ai = openai.OpenAI(api_key=OPENAI_API_KEY)

# Teste rápido
teste = client_ai.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role':'user','content':'Responda apenas: CONECTADO'}],
    max_tokens=10
)
print(f'\n✅ API OpenAI: {teste.choices[0].message.content}')
print(f'   Modelo: gpt-4o-mini | Custo: ~$0.001 por pergunta')

---
## ⚙️ BLOCO 6 — Motor do Chat
> Construímos 3 peças: **(1)** agregações pré-calculadas (o índice RAG), **(2)** roteador de intenção, **(3)** função de chat.
---

In [ ]:
# ============================================================
# CÉLULA 8 — Calcular agregações (o "índice" do RAG)
# Estratégia: calcular uma vez, reutilizar em cada pergunta.
# ============================================================
print('⚙️ Calculando agregações...')

_uf = df.groupby('uf_sg').agg(exist=('qt_exist','sum'), sus=('qt_sus','sum')).sort_values('exist', ascending=False)
_uf['taxa_sus'] = (_uf['sus'] / _uf['exist'] * 100).round(1)

_tipo = df.groupby('DS_TP_LEITO').agg(exist=('qt_exist','sum'), sus=('qt_sus','sum')).sort_values('exist', ascending=False)
_tipo['taxa_sus'] = (_tipo['sus'] / _tipo['exist'] * 100).round(1)

_df_uti   = df[df['DS_CO_LEITO'].str.contains('UTI', na=False)]
_uti_tipo = _df_uti.groupby('DS_CO_LEITO').agg(exist=('qt_exist','sum'), sus=('qt_sus','sum')).sort_values('exist', ascending=False)
_uti_tipo['taxa_sus'] = (_uti_tipo['sus'] / _uti_tipo['exist'] * 100).round(1)
_uti_uf   = _df_uti.groupby('uf_sg').agg(exist=('qt_exist','sum'), sus=('qt_sus','sum')).sort_values('exist', ascending=False)
_uti_uf['taxa_sus'] = (_uti_uf['sus'] / _uti_uf['exist'] * 100).round(1)

_reg = df.groupby('regiao').agg(exist=('qt_exist','sum'), sus=('qt_sus','sum'))
_reg['taxa_sus'] = (_reg['sus'] / _reg['exist'] * 100).round(1)

_esp = df.groupby('DS_CO_LEITO').agg(exist=('qt_exist','sum'), sus=('qt_sus','sum')).sort_values('exist', ascending=False).head(50)
_esp['taxa_sus'] = (_esp['sus'] / _esp['exist'] * 100).round(1)

_lt_e  = int(df['qt_exist'].sum())
_lt_s  = int(df['qt_sus'].sum())
_lt_n  = _lt_e - _lt_s
_uti_e = int(_df_uti['qt_exist'].sum())
_uti_s = int(_df_uti['qt_sus'].sum())

_RESUMO = f"""DATASET: CNES Leitos Hospitalares Brasil, Junho 2025
TOTAL: {len(df):,} registros | {_lt_e:,} leitos | {_lt_s:,} SUS ({_lt_s/_lt_e*100:.1f}%) | {_lt_n:,} não-SUS
UTI: {_uti_e:,} existentes | {_uti_s:,} SUS ({_uti_s/_uti_e*100:.1f}%)
UFs: {df['uf_sg'].nunique()} | TIPOS: {df['DS_TP_LEITO'].nunique()} | ESPECIALIDADES: {df['DS_CO_LEITO'].nunique()}"""

print(f'✅ Agregações prontas!')
print(f'   📋 Tipos de leito : {len(_tipo)}')
print(f'   🗺️  UFs            : {len(_uf)}')
print(f'   🔬 UTI tipos      : {len(_uti_tipo)}')
print(f'   🌍 Regiões        : {len(_reg)}')
print(f'   📊 Especialidades : {len(_esp)} (top 50)')

In [ ]:
# ============================================================
# CÉLULA 9 — Roteador + função de chat
# ============================================================

# ── ROTEADOR: lê a pergunta → escolhe tabelas relevantes ──
def _get_contexto(pergunta: str) -> str:
    p = pergunta.lower()
    blocos = [_RESUMO, f"\nLEITOS POR TIPO:\n{_tipo.to_string()}"]

    kw_uf = ['estado','uf','sp','rj','mg','rs','ba','pr','sc','go','df',
             'ac','am','pa','ma','ce','pe','pi','rn','pb','al','se','ap',
             'ro','rr','to','ms','mt','es','maior','menor','top','ranking','compare','comparar']
    if any(x in p for x in kw_uf):
        blocos.append(f"\nLEITOS POR UF:\n{_uf.to_string()}")

    if any(x in p for x in ['uti','intensiv','crítico','critico','complementar']):
        blocos.append(f"\nUTI POR TIPO:\n{_uti_tipo.to_string()}")
        blocos.append(f"\nUTI POR UF:\n{_uti_uf.to_string()}")

    if any(x in p for x in ['regiao','região','norte','nordeste','sudeste','sul','centro-oeste','centro']):
        blocos.append(f"\nLEITOS POR REGIÃO:\n{_reg.to_string()}")

    if any(x in p for x in ['especialidade','obstet','pediatr','cirurg','clínic','clinic',
                              'psiqui','oncol','neonat','hospital dia']):
        blocos.append(f"\nTOP 50 ESPECIALIDADES:\n{_esp.to_string()}")

    if len(blocos) == 2:
        blocos.append(f"\nLEITOS POR UF:\n{_uf.to_string()}")
        blocos.append(f"\nLEITOS POR REGIÃO:\n{_reg.to_string()}")

    return "\n".join(blocos)


# ── FUNÇÃO DE CHAT: une roteador + histórico + API ────────
_SYSTEM_PROMPT = """Você é um especialista em saúde pública brasileira com profundo conhecimento
do CNES (Cadastro Nacional de Estabelecimentos de Saúde) e do SUS.

INSTRUÇÕES:
1. Responda APENAS com base nos dados fornecidos no contexto
2. Cite números exatos — nunca estime sem avisar
3. Ao comparar UFs ou regiões, mostre os valores lado a lado
4. Se os dados não forem suficientes para responder, diga claramente
5. Responda em português, de forma direta e objetiva
6. Responda a pergunta primeiro, explique depois
7. Quando relevante, contextualize no sistema de saúde brasileiro
"""

def chat_leitos(pergunta: str, historico: list = []) -> str:
    contexto = _get_contexto(pergunta)
    msgs = [{'role': 'system', 'content': _SYSTEM_PROMPT}]
    for p_ant, r_ant in historico[-3:]:
        msgs.append({'role': 'user',      'content': p_ant})
        msgs.append({'role': 'assistant', 'content': r_ant})
    msgs.append({'role': 'user', 'content': f"DADOS CNES:\n{contexto}\n\nPERGUNTA: {pergunta}"})
    resp = client_ai.chat.completions.create(
        model='gpt-4o-mini', messages=msgs, temperature=0.1, max_tokens=700
    )
    return resp.choices[0].message.content


# Teste rápido
print('🧪 TESTE DO CHAT')
print('=' * 55)
pergunta_teste = 'Qual o total de leitos UTI SUS no Brasil e quais os estados com mais UTI?'
print(f'Pergunta: {pergunta_teste}\n')
print(chat_leitos(pergunta_teste))

---
## 💬 BLOCO 7 — Interface Gradio
> 🔴 **Execute APENAS após os Blocos 1, 5 e 6.**
> Se aparecer `name 'chat_leitos' is not defined`: **Ambiente de execução → Executar tudo** (`Ctrl+F9`)
---

In [ ]:
# ============================================================
# CÉLULA 10 — Interface Gradio (célula completa e autônoma)
# ============================================================
import gradio as gr

hist_gradio = []

def chat_gradio(pergunta: str, historico_visual: list):
    if not pergunta.strip():
        return historico_visual, ''
    try:
        resposta = chat_leitos(pergunta, hist_gradio)
        hist_gradio.append((pergunta, resposta))
    except Exception as e:
        resposta = f'❌ Erro: {str(e)[:300]}\n⚠️ Execute: Ambiente de execução → Executar tudo (Ctrl+F9)'
    return historico_visual + [(pergunta, resposta)], ''

def limpar_chat():
    global hist_gradio
    hist_gradio = []
    return [], ''

def btn_pergunta(texto):
    return lambda h: chat_gradio(texto, h)

with gr.Blocks(title='Chat Leitos SUS — IDASH 2026', theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div style='text-align:center;padding:20px;
                background:linear-gradient(135deg,#0f2942,#27ae60);
                border-radius:12px;margin-bottom:16px'>
        <h2 style='color:white;margin:0'>🏥 Assistente de Leitos Hospitalares</h2>
        <p style='color:#d5f5e3;margin:6px 0 0;font-size:13px'>
            CNES Jun/2025 · GPT-4o-mini · IDASH 2026
        </p>
    </div>""")

    chatbot = gr.Chatbot(height=420, show_label=False, bubble_full_width=False)

    with gr.Row():
        txt = gr.Textbox(
            placeholder='Pergunte sobre leitos hospitalares no Brasil...',
            show_label=False, scale=5, container=False
        )
        btn_enviar = gr.Button('Enviar', variant='primary', scale=1)

    btn_limpar = gr.Button('🗑️ Limpar conversa', size='sm')
    gr.Markdown('**💡 Perguntas rápidas:**')

    with gr.Row():
        b1 = gr.Button('🏥 UTI total no Brasil',  size='sm')
        b2 = gr.Button('🏆 Top 5 estados SUS',    size='sm')
        b3 = gr.Button('👶 Leitos obstétricos',   size='sm')
    with gr.Row():
        b4 = gr.Button('🌿 Região Norte',         size='sm')
        b5 = gr.Button('🔬 Tipos de UTI',         size='sm')
        b6 = gr.Button('📋 Resumo executivo',     size='sm')

    gr.HTML("""<details style='margin-top:12px'>
    <summary style='cursor:pointer;color:#7f8c8d;font-size:12px'>ℹ️ Como funciona (RAG)</summary>
    <p style='font-size:12px;color:#95a5a6;margin:8px 0 0'>
    Cada pergunta ativa um roteador que seleciona apenas as tabelas relevantes dos dados
    pré-calculados (UF, tipo, UTI, região, especialidade). Esse contexto é enviado ao
    GPT-4o-mini junto com a pergunta — nenhum dado bruto sai do Colab.
    </p></details>""")

    btn_enviar.click(chat_gradio, [txt, chatbot], [chatbot, txt])
    txt.submit(chat_gradio,       [txt, chatbot], [chatbot, txt])
    btn_limpar.click(limpar_chat, [], [chatbot, txt])

    b1.click(btn_pergunta('Quantos leitos UTI existem no Brasil? Quantos são SUS?'),              [chatbot], [chatbot, txt])
    b2.click(btn_pergunta('Quais os 5 estados com maior % de leitos SUS? E os 5 menores?'),       [chatbot], [chatbot, txt])
    b3.click(btn_pergunta('Compare leitos obstétricos entre SP, MG, RJ e BA.'),                   [chatbot], [chatbot, txt])
    b4.click(btn_pergunta('Qual a situação dos leitos na região Norte? Compare com o Sudeste.'),  [chatbot], [chatbot, txt])
    b5.click(btn_pergunta('Quais os tipos de UTI existentes e quantos leitos tem cada um?'),      [chatbot], [chatbot, txt])
    b6.click(btn_pergunta('Faça um resumo executivo dos leitos hospitalares no Brasil, destacando desigualdades regionais.'), [chatbot], [chatbot, txt])

demo.launch(share=False, debug=False, quiet=True)
print('\n✅ Chat rodando! Acesse o link acima.')

---
## 📤 BLOCO 8 — Exportar Resultados
> Gera um `.txt` para o **NotebookLM** — qualquer pessoa faz perguntas sem precisar de Python ou API Key.
---

In [ ]:
# ============================================================
# CÉLULA 11 — Exportar análise para NotebookLM
# ============================================================
from google.colab import files

linhas = [
    '# ANÁLISE CNES — Leitos Hospitalares Brasil',
    '## Competência: Junho 2025 | Fonte: DATASUS/CNES | IDASH 2026',
    '',
    '## 1. RESUMO GERAL',
    f'- Registros: {len(df):,}',
    f'- Leitos existentes: {_lt_e:,}',
    f'- Leitos SUS: {_lt_s:,} ({_lt_s/_lt_e*100:.1f}%)',
    f'- Leitos não-SUS: {_lt_n:,}',
    f'- UTI existentes: {_uti_e:,}',
    f'- UTI SUS: {_uti_s:,} ({_uti_s/_uti_e*100:.1f}%)',
    '',
    '## 2. LEITOS POR TIPO',
]
for idx, row in _tipo.iterrows():
    linhas.append(f'- {idx}: {int(row["exist"]):,} leitos ({row["taxa_sus"]}% SUS)')

linhas += ['', '## 3. LEITOS POR UF']
for uf, row in _uf.iterrows():
    uti_r    = _uti_uf[_uti_uf.index == uf]
    uti_info = f' | UTI: {int(uti_r["exist"].values[0]):,} ({uti_r["taxa_sus"].values[0]}% SUS)' if len(uti_r) > 0 else ''
    linhas.append(f'- {uf}: {int(row["exist"]):,} leitos ({row["taxa_sus"]}% SUS){uti_info}')

linhas += ['', '## 4. LEITOS POR REGIÃO']
for reg, row in _reg.iterrows():
    linhas.append(f'- {reg}: {int(row["exist"]):,} leitos ({row["taxa_sus"]}% SUS)')

linhas += [
    '', '## 5. PRINCIPAIS ACHADOS',
    f'- {_lt_e:,} leitos no Brasil — {_lt_s/_lt_e*100:.1f}% SUS',
    '- Tipo CLÍNICO é o maior; COMPLEMENTAR inclui todas as UTIs',
    '- Sudeste concentra maior volume; Norte/Nordeste têm maior % SUS',
    '- DF tem menor taxa SUS; AC e RR as maiores',
    f'- {_uti_e:,} leitos UTI — {_uti_s/_uti_e*100:.1f}% SUS',
]

texto = '\n'.join(linhas)
with open('cnes_leitos_notebooklm.txt', 'w', encoding='utf-8') as f:
    f.write(texto)

print(f'✅ Arquivo gerado: cnes_leitos_notebooklm.txt ({len(texto):,} chars)')
print()
print('📌 Como usar no NotebookLM:')
print('   1. notebooklm.google.com')
print('   2. Crie um novo notebook')
print('   3. Add source → Upload → cnes_leitos_notebooklm.txt')
print('   4. Faça perguntas em português!')

files.download('cnes_leitos_notebooklm.txt')

---
## 🎓 Recapitulação e Próximos Passos

### ✅ O que construímos
```
CnesLeitosBR.csv
  │
  ├── Bloco 1 → 5 bibliotecas + configuração
  ├── Bloco 2 → Limpeza + uf_sg + regiao
  ├── Bloco 3 → EDA: resumo + 3 gráficos
  ├── Bloco 3B→ Validação SQL (fora do Colab)
  ├── Bloco 4 → Conceito RAG vs Fine-tuning
  ├── Bloco 5 → API OpenAI conectada
  ├── Bloco 6 → Agregações + roteador + chat_leitos()
  ├── Bloco 7 → Interface Gradio navegável
  └── Bloco 8 → Export NotebookLM
```

### 🚀 Próximos passos
- **Nível 1:** Trocar o CSV por outro dataset (SINAN, SIM, e-SUS)
- **Nível 2:** Embeddings + FAISS (RAG semântico) · Text-to-SQL
- **Nível 3:** Fine-tuning · Agente multi-fonte · Deploy em produção

> 💡 **Lição central:** A IA é tão boa quanto o contexto que você dá a ela.  
> **Dados limpos + Contexto relevante + Prompt certo = Produto que funciona.**
